In [9]:
import sys
import os
import importlib.util

# 1. Define paths
# Root is the folder CONTAINING the syllable_generation folder
project_root = "/home/fsp585/Desktop/FiberphotometryCode"
datamodel_path = os.path.join(project_root, "syllable_generation/datamodel.py")

# 2. Force-load the module into sys.modules
# This bypasses the need for __init__.py or perfect sys.path resolution
spec = importlib.util.spec_from_file_location("syllable_generation.datamodel", datamodel_path)
dm_module = importlib.util.module_from_spec(spec)
sys.modules["syllable_generation.datamodel"] = dm_module
spec.loader.exec_module(dm_module)

print(f"✅ Forced mapping: 'syllable_generation.datamodel' -> {datamodel_path}")

# 3. Add to path for other relative imports
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Test import
import syllable_generation.datamodel as dm
print("Test import successful. You can now load the pickle.")

✅ Forced mapping: 'syllable_generation.datamodel' -> /home/fsp585/Desktop/FiberphotometryCode/syllable_generation/datamodel.py
Test import successful. You can now load the pickle.


In [12]:
import pickle

PKL_PATH = 'jonathan_ifdn_sessions_w_sylls.pkl'

with open(PKL_PATH, 'rb') as f:
    sessions = pickle.load(f)

print(f"Successfully loaded {len(sessions)} sessions.")
s = sessions[0]

# Display columns as requested
print("\n--- all_data_df columns ---")
print(list(s.all_data_df.columns))

Successfully loaded 19 sessions.

--- all_data_df columns ---
['frame_index', 'name', 'centroid_x', 'centroid_y', 'heading', 'angular_velocity', 'velocity_px_s', 'syllable', 'group', 'onset', 'SystemTimestamp', 'signal_0', 'signal_1', 'signal_2', 'signal_3', 'signal_0_dff', 'signal_0_zscore', 'signal_0_zscore_datta', 'signal_1_dff', 'signal_1_zscore', 'signal_1_zscore_datta', 'signal_2_dff', 'signal_2_zscore', 'signal_2_zscore_datta', 'signal_3_dff', 'signal_3_zscore', 'signal_3_zscore_datta']


In [13]:
import os
import json
import pandas as pd

# 1. Config
# Using the session list loaded in the previous cells
target_pkl = PKL_PATH  # 'jonathan_amph_sessions_v2_w_sylls.pkl'
output_txt = target_pkl.replace('.pkl', '.txt')

def get_inspection_report(sessions, output_file):
    if not sessions:
        print("No sessions found in list.")
        return

    s = sessions[0]
    df = getattr(s, 'all_data_df', None)
    
    with open(output_file, 'w') as f:
        f.write(f"INSPECTION REPORT: {target_pkl}\n")
        f.write("="*60 + "\n\n")

        # --- Section 1: Session Metadata ---
        f.write("1. SESSION METADATA (Sample Session 0)\n")
        f.write("-" * 40 + "\n")
        # Filter out heavy exclusion lists for the text report
        clean_meta = {k: v for k, v in s.metadata.items() if 'exclusion' not in k.lower()}
        f.write(json.dumps(clean_meta, indent=4, default=str))
        f.write("\n\n")

        # --- Section 2: Signal Metadata ---
        f.write("2. SIGNAL METADATA\n")
        f.write("-" * 40 + "\n")
        if hasattr(s, 'signal_meta'):
            f.write(json.dumps(s.signal_meta, indent=4, default=str))
        else:
            f.write("Attribute 'signal_meta' not found.")
        f.write("\n\n")

        # --- Section 3: DataFrame Structure ---
        f.write("3. ALL_DATA_DF COLUMNS & TYPES\n")
        f.write("-" * 40 + "\n")
        if df is not None:
            # Create a summary table of columns and their types
            col_info = pd.DataFrame({
                'Type': df.dtypes.astype(str),
                'Non-Null Count': df.count(),
                'NaN Count': df.isna().sum()
            })
            f.write(col_info.to_string())
            f.write(f"\n\nTotal Rows: {len(df)}")
            
            # --- Section 4: Weaving Diagnostic (Syllable Coverage) ---
            f.write("\n\n4. SYLLABLE WEAVING DIAGNOSTIC\n")
            f.write("-" * 40 + "\n")
            if 'syllable' in df.columns:
                valid_sylls = df['syllable'].notna().sum()
                coverage = (valid_sylls / len(df)) * 100
                f.write(f"Syllable Coverage: {valid_sylls} / {len(df)} frames ({coverage:.2f}%)\n")
                f.write(f"Unique Syllables found: {df['syllable'].nunique()}\n")
                f.write("\nTop 10 Syllables by Frequency:\n")
                f.write(df['syllable'].value_counts().head(10).to_string())
            else:
                f.write("'syllable' column missing from dataframe.")
        else:
            f.write("Attribute 'all_data_df' not found.")

    print(f"✅ Inspection report saved to: {os.path.abspath(output_file)}")

# Run the function
get_inspection_report(sessions, output_txt)

✅ Inspection report saved to: /home/fsp585/Desktop/FiberphotometryCode/syllable_generation/data/jonathan_ifdn_sessions_w_sylls.txt
